In [1]:
import glob
import string
import emoji
import os
import pandas as pd

In [2]:
# CHANGE ACCORDINGLY: Define your mapping of CSV file names to game titles
CONSOLIDATED_METACRITIC_BASE_DIR = "metacritic_reviews/csv_files"
OUTPUT_METACRITIC_DIR = "metacritic_reviews/updated_csv_files"
CONSOLIDATED_STEAM_BASE_DIR = "steam_reviews/csv_files"
OUTPUT_STEAM_DIR = "steam_reviews/updated_csv_files"
CONSOLIDATED_REDDIT_BASE_DIR = "reddit_reviews/csv_files"
OUTPUT_REDDIT_DIR = "reddit_reviews/updated_csv_files"

# CHANGE ACCORDINGLY: Define your mapping of CSV file names to game titles
# The key is the file name
metacritc_game_titles = {
    'cyberpunk-2077': 'Cyberpunk 2077',
    'sea-of-thieves': 'Sea of Thieves',
    'fallout-76': 'Fallout 76',
    'total-war-rome-ii': 'Total War: Rome 2',
    'battlefield-2042': 'Battlefield 2042',
    'days-gone': 'Days Gone',
    'wildfrost': 'Wildfrost',
    'final-fantasy-xiv-online': 'Final Fantasy 14',
    'warhammer-40000-darktide': 'Warhammer 40k: Darktide',
    'wasteland-3': 'Wasteland 3'
}

reddit_game_titles = {
    'Cyberpunk_2077': 'Cyberpunk 2077',
    'Sea_of_Thieves': 'Sea of Thieves',
    'Fallout_76': 'Fallout 76',
    'Total_War_Rome_2': 'Total War: Rome 2',
    'Battlefield_2042': 'Battlefield 2042',
    'Days_Gone': 'Days Gone',
    'Wildfrost': 'Wildfrost',
    'Final_Fantasy_14': 'Final Fantasy 14',
    'Warhammer_40k__Darktide': 'Warhammer 40k: Darktide',
    'Wasteland_3': 'Wasteland 3'
}

steam_game_titles = {
    'cyberpunk_2077_reviews': 'Cyberpunk 2077',
    'sea_of_thieves_reviews': 'Sea of Thieves',
    'Fallout76_Steam': 'Fallout 76',
    'total_war_reviews': 'Total War: Rome 2',
    'battlefield_2042_reviews': 'Battlefield 2042',
    'days_gone_reviews': 'Days Gone',
    'wildfrost_reviews': 'Wildfrost',
    'final_fantasy_14_reviews': 'Final Fantasy 14',
    'warhammer_reviews': 'Warhammer 40k: Darktide',
    'wasteland_3_reviews': 'Wasteland 3'
}

In [29]:
def data_processing(base_dir, output_dir, game_titles, platform):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Get all CSV files
    csv_files = glob.glob(os.path.join(base_dir, "*.csv"))
    
    for csv_file in csv_files:
        print(f"Processing {csv_file}")
        
        file_name = os.path.splitext(os.path.basename(csv_file))[0]
        df = pd.read_csv(csv_file)
        
        df.columns = df.columns.str.strip()  # Always strip whitespace from columns first

        # Platform-specific processing FIRST
        if platform == "Metacritic":
            df = df.rename(columns={
                'platform': 'gaming_platform',
                'date': 'timestamp_updated_date',
                'review': 'review_text'
            })
            df['timestamp_updated_date'] = pd.to_datetime(df['timestamp_updated_date'], errors='coerce').dt.strftime("%Y-%m-%d")
            df['score'] = df['score'].replace('tbd', pd.NA)

            output_filename = f"updated_{file_name}.csv"

        elif platform == "Reddit":
            if 'Comment Date' in df.columns:
                df['Comment Date'] = df['Comment Date'].astype(str).str.strip()
                df['Comment Date'] = pd.to_datetime(df['Comment Date'], errors='coerce')
                df['timestamp_updated_date'] = df['Comment Date'].dt.date

                if df['Comment Date'].isna().sum() > 0:
                    print(f"Warning: Some 'Comment Date' values could not be converted in {file_name}.csv!")
            else:
                print(f"Warning: 'Comment Date' column not found in {file_name}.csv")

            output_filename = f"updated_{file_name}.csv"

        elif platform == "Steam":
            if 'timestamp_updated' in df.columns:
                df['timestamp_updated'] = pd.to_datetime(df['timestamp_updated'])
                df['timestamp_updated_date'] = df['timestamp_updated'].dt.date
            else:
                print(f"Warning: 'timestamp_updated' column not found in {file_name}.csv")

            output_filename = f"updated_{file_name}.csv"

        else:
            print(f"Warning: Unknown platform '{platform}'! Skipping file.")
            continue  # skip saving if platform is unknown

        # THEN assign platform and game columns
        df['platform'] = platform
        game_title = game_titles.get(file_name)

        if game_title:
            df['game'] = game_title
        else:
            print(f"Warning: No game title found for '{file_name}'!")
            
        if platform == "Steam":
            df = df[['author_steamid','review_text','votes_up', 'votes_funny', 'weighted_vote_score','platform', 'game', 'timestamp_updated_date']]
        elif platform == "Reddit":
            df = df[['Title', 'URL', 'Score', 'review_text', 'Subreddit', 'Comment Author', 'Level', 'platform', 'game', 'timestamp_updated_date']]

        # Save the cleaned dataframe
        output_path = os.path.join(output_dir, output_filename)
        df.to_csv(output_path, index=False)
        print(f"Saved updated file: {output_path}\n")


In [4]:
data_processing(CONSOLIDATED_METACRITIC_BASE_DIR,OUTPUT_METACRITIC_DIR,metacritc_game_titles,"Metacritic")

Processing metacritic_reviews/csv_files/wildfrost.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_wildfrost.csv

Processing metacritic_reviews/csv_files/warhammer-40000-darktide.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_warhammer-40000-darktide.csv

Processing metacritic_reviews/csv_files/days-gone.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_days-gone.csv

Processing metacritic_reviews/csv_files/wasteland-3.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_wasteland-3.csv

Processing metacritic_reviews/csv_files/fallout-76.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_fallout-76.csv

Processing metacritic_reviews/csv_files/final-fantasy-xiv-online.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_final-fantasy-xiv-online.csv

Processing metacritic_reviews/csv_files/total-war-rome-ii.csv
Saved updated file: metacritic_reviews/updated_csv_files/updated_tot

In [30]:
data_processing(CONSOLIDATED_STEAM_BASE_DIR,OUTPUT_STEAM_DIR,steam_game_titles,"Steam")

Processing steam_reviews/csv_files/wasteland_3_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_wasteland_3_reviews.csv

Processing steam_reviews/csv_files/warhammer_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_warhammer_reviews.csv

Processing steam_reviews/csv_files/wildfrost_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_wildfrost_reviews.csv

Processing steam_reviews/csv_files/sea_of_thieves_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_sea_of_thieves_reviews.csv

Processing steam_reviews/csv_files/days_gone_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_days_gone_reviews.csv

Processing steam_reviews/csv_files/battlefield_2042_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_battlefield_2042_reviews.csv

Processing steam_reviews/csv_files/cyberpunk_2077_reviews.csv
Saved updated file: steam_reviews/updated_csv_files/updated_cyberpunk_

In [25]:
data_processing(CONSOLIDATED_REDDIT_BASE_DIR,OUTPUT_REDDIT_DIR,reddit_game_titles,"Reddit")

Processing reddit_reviews/csv_files/Battlefield_2042.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Battlefield_2042.csv

Processing reddit_reviews/csv_files/Wildfrost.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Wildfrost.csv

Processing reddit_reviews/csv_files/Final_Fantasy_14.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Final_Fantasy_14.csv

Processing reddit_reviews/csv_files/Wasteland_3.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Wasteland_3.csv

Processing reddit_reviews/csv_files/Warhammer_40k__Darktide.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Warhammer_40k__Darktide.csv

Processing reddit_reviews/csv_files/Sea_of_Thieves.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Sea_of_Thieves.csv

Processing reddit_reviews/csv_files/Days_Gone.csv
Saved updated file: reddit_reviews/updated_csv_files/updated_Days_Gone.csv

Processing reddit_reviews/csv_files/Total_War_Ro

In [7]:
# Function to check the percentage of symbols in a string
def is_mostly_symbols(text, threshold=0.8):
    text = str(text)
    # Define what counts as a symbol (punctuation + other special characters)
    symbols = set(string.punctuation + '£$%^&*()_+=<>?{}[]|\\~`')

    # Count symbols and emojis
    symbol_count = sum(1 for char in text if char in symbols or emoji.is_emoji(char))

    # Calculate the percentage of symbols
    total_chars = len(text)
    if total_chars == 0:
        return False  # Handle empty strings

    symbol_ratio = symbol_count / total_chars

    # Return True if the symbol ratio exceeds the threshold
    return symbol_ratio > threshold


In [31]:
# CHANGE ACCORDINGLY: Edit to your base directory, might need to create additional folder 'Updated_CSV_files'
OUTPUT_METACRITIC_DIR = "metacritic_reviews/updated_csv_files"
OUTPUT_STEAM_DIR = "steam_reviews/updated_csv_files"
OUTPUT_REDDIT_DIR = "reddit_reviews/updated_csv_files"

def remove_symbols_blank_na_from_csv_files(updated_base_dir):
    # Ensure the updated directory exists
    os.makedirs(updated_base_dir, exist_ok=True)

    # Get all CSV files in the base directory
    csv_files = glob.glob(f"{updated_base_dir}/*.csv")

    for csv_file in csv_files:
        print(f"Processing {csv_file}")

        # Load the CSV file
        df = pd.read_csv(csv_file, lineterminator='\n')

        # Print number of NaNs BEFORE cleaning
        print(f"Before cleaning - NaN review_text rows: {df['review_text'].isna().sum()}")

        # Remove rows where review_text is mostly symbols, empty, NaN, or [deleted]
        updated_df = df[
            df['review_text'].notna() &  # keep only non-NaN
            df['review_text'].str.strip().ne('') &  # not empty after strip
            df['review_text'].str.strip().ne('[deleted]') &  # not [deleted]
            ~df['review_text'].apply(is_mostly_symbols)  # not mostly symbols
        ]

        # Print number of NaNs AFTER cleaning
        print(f"After cleaning - NaN review_text rows: {updated_df['review_text'].isna().sum()}")
        print(f"--------Updated Length of {csv_file}: {len(updated_df)}--------")

        # Save cleaned DataFrame (overwrite)
        output_file = os.path.join(csv_file)
        updated_df.to_csv(output_file, index=False)
        print(f"Saved updated file: {output_file}\n")

# remove_symbols_blank_na_from_csv_files(OUTPUT_METACRITIC_DIR)
# remove_symbols_blank_na_from_csv_files(OUTPUT_REDDIT_DIR)
remove_symbols_blank_na_from_csv_files(OUTPUT_STEAM_DIR)


Processing steam_reviews/updated_csv_files/updated_battlefield_2042_reviews.csv
Before cleaning - NaN review_text rows: 270
After cleaning - NaN review_text rows: 0
--------Updated Length of steam_reviews/updated_csv_files/updated_battlefield_2042_reviews.csv: 119074--------
Saved updated file: steam_reviews/updated_csv_files/updated_battlefield_2042_reviews.csv

Processing steam_reviews/updated_csv_files/updated_Fallout76_Steam.csv
Before cleaning - NaN review_text rows: 233
After cleaning - NaN review_text rows: 0
--------Updated Length of steam_reviews/updated_csv_files/updated_Fallout76_Steam.csv: 68900--------
Saved updated file: steam_reviews/updated_csv_files/updated_Fallout76_Steam.csv

Processing steam_reviews/updated_csv_files/updated_wildfrost_reviews.csv
Before cleaning - NaN review_text rows: 6
After cleaning - NaN review_text rows: 0
--------Updated Length of steam_reviews/updated_csv_files/updated_wildfrost_reviews.csv: 5192--------
Saved updated file: steam_reviews/upda

In [ ]:
def show_duplicates(updated_base_dir):
    # Ensure the updated directory exists
    os.makedirs(updated_base_dir, exist_ok=True)

    # Get all CSV files in the base directory
    csv_files = glob.glob(f"{updated_base_dir}/*.csv")
    csv_file = csv_files[0]
    for csv_file in csv_files:
        print(f"Processing {csv_file}")

        # Load the CSV file
        df = pd.read_csv(csv_file, lineterminator='\n')

        # Count total and duplicate rows BEFORE any cleaning
        total_rows = len(df)
        duplicate_rows = df.duplicated().sum()
        print(f"Total rows: {total_rows}")
        print(f"Duplicate rows: {duplicate_rows}")
        # print(df.duplicated())
        # duplicates = df[df.duplicated()]
        # print(duplicates)


show_duplicates(OUTPUT_METACRITIC_DIR)
show_duplicates(OUTPUT_REDDIT_DIR)
show_duplicates(OUTPUT_STEAM_DIR)

In [32]:
def remove_duplicates(updated_base_dir):
    # Ensure the updated directory exists
    os.makedirs(updated_base_dir, exist_ok=True)

    # Get all CSV files in the base directory
    csv_files = glob.glob(f"{updated_base_dir}/*.csv")
    
    for csv_file in csv_files:
        print(f"Processing {csv_file}")

        # Load the CSV file
        df = pd.read_csv(csv_file, lineterminator='\n')

        # Count total and duplicate rows BEFORE removal
        total_rows = len(df)
        duplicate_rows = df.duplicated().sum()
        print(f"Total rows: {total_rows}")
        print(f"Duplicate rows found: {duplicate_rows}")

        if duplicate_rows > 0:
            # Show duplicate rows
            duplicates = df[df.duplicated()]
            print(f"Sample duplicate rows:\n{duplicates.head()}\n")

            # Remove duplicates
            df = df.drop_duplicates()
            print(f"Rows after removing duplicates: {len(df)}")

            # Overwrite the original file
            df.to_csv(csv_file, index=False)
            print(f"Saved file without duplicates: {csv_file}\n")
        else:
            print("No duplicates found. No changes made.\n")


        
# remove_duplicates(OUTPUT_METACRITIC_DIR)
# remove_duplicates(OUTPUT_REDDIT_DIR)
remove_duplicates(OUTPUT_STEAM_DIR)

Processing steam_reviews/updated_csv_files/updated_battlefield_2042_reviews.csv
Total rows: 119074
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_Fallout76_Steam.csv
Total rows: 68900
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_wildfrost_reviews.csv
Total rows: 5192
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_warhammer_reviews.csv
Total rows: 75118
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_days_gone_reviews.csv
Total rows: 42270
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_wasteland_3_reviews.csv
Total rows: 13111
Duplicate rows found: 0
No duplicates found. No changes made.

Processing steam_reviews/updated_csv_files/updated_total_war_reviews.

In [ ]:
# Print updated length of datasets to compare

# Iterate over each CSV file
def sanitycheck(DIR):
    csv_files = glob.glob(f"{DIR}/*.csv")
    for csv_file in csv_files:

        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(csv_file,lineterminator='\n')

        print(f"--------Updated Length of {csv_file}: {len(df)}--------")

In [ ]:
# BEFORE DATA CLEANING
# Use glob to get all CSV files in the directory
sanitycheck(CONSOLIDATED_METACRITIC_BASE_DIR)
sanitycheck(CONSOLIDATED_REDDIT_BASE_DIR)
sanitycheck(CONSOLIDATED_STEAM_BASE_DIR)

--------Updated Length of metacritic_reviews/csv_files/wildfrost.csv: 52--------
--------Updated Length of metacritic_reviews/csv_files/warhammer-40000-darktide.csv: 199--------
--------Updated Length of metacritic_reviews/csv_files/days-gone.csv: 4561--------
--------Updated Length of metacritic_reviews/csv_files/wasteland-3.csv: 460--------
--------Updated Length of metacritic_reviews/csv_files/fallout-76.csv: 2665--------
--------Updated Length of metacritic_reviews/csv_files/final-fantasy-xiv-online.csv: 202--------
--------Updated Length of metacritic_reviews/csv_files/total-war-rome-ii.csv: 121--------
--------Updated Length of metacritic_reviews/csv_files/cyberpunk-2077.csv: 12593--------
--------Updated Length of metacritic_reviews/csv_files/battlefield-2042.csv: 3845--------
--------Updated Length of metacritic_reviews/csv_files/sea-of-thieves.csv: 1229--------


In [ ]:
# AFTER DATA CLEANING
sanitycheck(OUTPUT_METACRITIC_DIR)
sanitycheck(OUTPUT_REDDIT_DIR)
sanitycheck(OUTPUT_STEAM_DIR)

--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_fallout-76.csv: 2661--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_total-war-rome-ii.csv: 121--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_wasteland-3.csv: 460--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_final-fantasy-xiv-online.csv: 201--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_wildfrost.csv: 52--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_warhammer-40000-darktide.csv: 199--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_days-gone.csv: 4554--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_cyberpunk-2077.csv: 12571--------
--------Updated Length of metacritic_reviews/Updated_CSV_files/updated_battlefield-2042.csv: 3843--------
--------Updated Length of metacritic_reviews/Updated_CSV_file